# Click Reaction AI Exploration

이 노트북은 `data/raw/click_reaction_literature.csv`를 정리한 뒤, cleaned dataset을 검증하고 feature table, 군집/예측 분석 결과를 확인하기 위한 시작점입니다.

In [1]:
from pathlib import Path
import sys


def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "src").is_dir() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate project root from current working directory.")


PROJECT_ROOT = find_project_root()
SRC_DIR = PROJECT_ROOT / "src"

for path in [PROJECT_ROOT, SRC_DIR]:
    path_str = str(path)
    if path_str not in sys.path:
        sys.path.insert(0, path_str)

RAW_DATA = PROJECT_ROOT / "data" / "raw" / "click_reaction_literature.csv"
CLEANED_DATA = PROJECT_ROOT / "data" / "processed" / "click_reaction_literature_cleaned.csv"
CLEANING_REPORT = PROJECT_ROOT / "data" / "processed" / "literature_cleaning_report.json"

if not RAW_DATA.exists():
    raise FileNotFoundError(f"Expected literature dataset at {RAW_DATA}")

RAW_DATA


WindowsPath('C:/Users/theripus/OneDrive - 전남과학고등학교/문서/정보/project-aipersonal/data/raw/click_reaction_literature.csv')

In [2]:
from src.clean_literature_dataset import clean_file

cleaning_report = clean_file(RAW_DATA, CLEANED_DATA, CLEANING_REPORT)
cleaning_report


{'rows': 89,
 'valid_after_cleaning': True,
 'errors': [],
 'warnings': ['13 row(s) are not primary_checked. Treat analysis as exploratory.'],
 'rows_with_numeric_k': 26,
 'rows_with_numeric_yield': 54,
 'rows_with_numeric_Ea': 0,
 'reaction_type_counts': {'CuAAC': 53, 'SPAAC': 34, 'thermal_Huisgen': 2},
 'verification_status_counts': {'primary_checked': 76,
  'needs_primary_check': 13},
 'recommendation': 'Good for exploratory workflow and yield/condition summaries; add more primary-checked k_value rows before making strong claims about reaction-rate prediction.',
 'input': 'C:\\Users\\theripus\\OneDrive - 전남과학고등학교\\문서\\정보\\project-aipersonal\\data\\raw\\click_reaction_literature.csv',
 'output': 'C:\\Users\\theripus\\OneDrive - 전남과학고등학교\\문서\\정보\\project-aipersonal\\data\\processed\\click_reaction_literature_cleaned.csv'}

In [3]:
import pandas as pd

df = pd.read_csv(CLEANED_DATA)
df.head()


,record_id,reaction_type,azide,alkyne,catalyst,catalyst_present,solvent,solvent_class,temperature_K,pH,...,alkyne_family_original,k_value_original,temperature_K_original,pH_original,Ea_kJ_mol_original,yield_percent_original,ring_strain_level_original,fluorine_count_original,fused_aromatic_count_original,heteroatom_in_ring_original
0,rec001,SPAAC,benzyl azide,cyclooctyne compound 12,none,0,CD3CN,organic,NaN,NaN,...,cyclooctyne,4.3 × 10-3,NaN,NaN,NaN,NaN,strained cyclooctyne,1.0,NaN,NaN
1,rec002,SPAAC,benzyl azide,free acid of cyclooctyne compound 1,none,0,CD3CN,organic,NaN,NaN,...,cyclooctyne,2.4 × 10-3,NaN,NaN,NaN,NaN,strained cyclooctyne,NaN,NaN,NaN
2,rec003,SPAAC,benzyl azide,cyclooctyne compound 8,none,0,CD3CN,organic,NaN,NaN,...,cyclooctyne,1.3 × 10-3,NaN,NaN,NaN,NaN,strained cyclooctyne,0.0,NaN,NaN
3,rec004,SPAAC,benzyl azide,cyclooctyne compound 13,none,0,CD3CN,organic,NaN,NaN,...,cyclooctyne,1.2 × 10-3,NaN,NaN,NaN,NaN,strained cyclooctyne,0.0,NaN,NaN
4,rec005,SPAAC,benzyl azide,DIFO,none,0,NaN,unknown,NaN,NaN,...,difluorinated cyclooctyne,7.6 × 10-2,NaN,NaN,NaN,NaN,strained cyclooctyne,2.0,0.0,NaN


In [4]:
from src.validate_dataset import validate_dataframe

errors, warnings = validate_dataframe(df)
errors, warnings


([], ['13 row(s) are not primary_checked. Treat analysis as exploratory.'])

In [5]:
from src.build_features import build_feature_table

features = build_feature_table(df)
features.head()


,record_id,catalyst_present,catalyst_present_missing,temperature_K,temperature_K_missing,pH,pH_missing,Ea_kJ_mol,Ea_kJ_mol_missing,yield_percent,...,alkyne_family_bicyclononyne,alkyne_family_cyclooctyne,alkyne_family_dibenzocyclooctyne,alkyne_family_fluorinated_cyclooctyne,alkyne_family_unknown,log10_k,k_value,alkyne,rate_label,verification_status
0,rec001,0,0,25.0,1,7.0,1,0.0,1,0.0,...,False,True,False,False,False,-2.366532,0.0043,cyclooctyne compound 12,,primary_checked
1,rec002,0,0,25.0,1,7.0,1,0.0,1,0.0,...,False,True,False,False,False,-2.619789,0.0024,free acid of cyclooctyne compound 1,,primary_checked
2,rec003,0,0,25.0,1,7.0,1,0.0,1,0.0,...,False,True,False,False,False,-2.886057,0.0013,cyclooctyne compound 8,,primary_checked
3,rec004,0,0,25.0,1,7.0,1,0.0,1,0.0,...,False,True,False,False,False,-2.920819,0.0012,cyclooctyne compound 13,,primary_checked
4,rec005,0,0,25.0,1,7.0,1,0.0,1,0.0,...,False,False,False,True,False,-1.119186,0.0760,DIFO,,primary_checked
